In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import urllib.request


In [13]:
url = 'https://www.sigma-cap.com/main/fl_market_page.overview?u_sess='

# Fetch the page
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
try:
    with urllib.request.urlopen(req) as response:
        html = response.read()
except Exception as e:
    print(f"Failed to fetch: {e}")
    html = b''

soup = BeautifulSoup(html, 'html.parser', from_encoding='windows-1256')

In [14]:
# 1 & 2. Institutional vs. Individual / Foreign vs. Domestic
tables = soup.find_all('table', class_='marginTable')

inst_foreign_table = None
for t in tables:
    if 'Egyptians' in t.text and 'Foreigners' in t.text:
        inst_foreign_table = t
        break

if inst_foreign_table:
    try:
        from io import StringIO
        df_inst_foreign = pd.read_html(StringIO(str(inst_foreign_table)))[0]
    except Exception:
        df_inst_foreign = pd.read_html(str(inst_foreign_table))[0]
    
    if isinstance(df_inst_foreign.columns, pd.MultiIndex):
        df_inst_foreign.columns = ['_'.join(col).strip() for col in df_inst_foreign.columns.values]
    
    print("--- Institutional vs. Individual / Foreign vs. Domestic ---")
    display(df_inst_foreign)
else:
    print("Could not find the Institutional/Foreign table.")

--- Institutional vs. Individual / Foreign vs. Domestic ---


,0,1,2,3,4,5,6,7,8
0,Trade,Egyptians,Egyptians,Arabs,Arabs,Foreigners,Foreigners,Totals,Totals
1,Trade,Ret.,Inst.,Ret.,Inst.,Ret.,Inst.,Ret.,Inst.
2,Buy (M),7470.79,1167.96,209.80,65.28,16.14,690.24,7696.73,1923.48
3,Sell (M),7714.76,1115.39,172.75,156.38,4.19,456.75,7891.69,1728.52
4,Net (M),-243.97,52.58,37.05,-91.10,11.95,233.49,-194.97,194.97
5,Net (M),-191.39,-191.39,-54.05,-54.05,245.44,245.44,NaN,NaN
6,Turnover (M),15185.54,2283.35,382.54,221.66,20.33,1146.99,15588.42,3652.00
7,Trading %,78.93,11.87,1.99,1.15,0.11,5.96,81.02,18.98
8,Totals,"17,468.89 (90.79%)","17,468.89 (90.79%)",604.20 (3.14%),604.20 (3.14%),"1,167.32 (6.07%)","1,167.32 (6.07%)",NaN,NaN


In [15]:
# 3. Block Trades
block_trades_table = None
for t in soup.find_all('table'):
    if 'Share Name' in t.text and 'Value EGP' in t.text:
        block_trades_table = t
        break

if block_trades_table:
    try:
        from io import StringIO
        df_block_trades = pd.read_html(StringIO(str(block_trades_table)))[0]
    except Exception:
        df_block_trades = pd.read_html(str(block_trades_table))[0]
    
    if df_block_trades.iloc[0].astype(str).str.contains('Share Name').any():
        df_block_trades.columns = df_block_trades.iloc[0]
        df_block_trades = df_block_trades[1:].reset_index(drop=True)
        
    if str(df_block_trades.columns[0]).startswith('Unnamed') or pd.isna(df_block_trades.columns[0]) or str(df_block_trades.columns[0]).strip() == '':
        df_block_trades = df_block_trades.drop(df_block_trades.columns[0], axis=1)
        
    print("\n--- Block Trades ---")
    display(df_block_trades)
else:
    print("Could not find the Block Trades table.")


--- Block Trades ---


,Share Name,Qty.,Value EGP,Count
0,Qalaa For Financial Investments,112695356,590411756,280
1,T M G Holding,1472962,143072344,48
2,Commercial International Bank-Egypt (Cib),826635,110701907,52
3,Korra,29327054,99049962,50
4,Heliopolis Housing,11050304,75434526,32
5,Beltone Holding,19523242,60253674,30
6,Abou Kir Fertilizers,656392,46609089,17
7,Memphis Pharmaceuticals,192500,46166650,3
8,Orascom Investment Holding,29111487,41119511,23
9,Six Of October Development & Investment (Sodic),1503417,40249098,18


In [16]:
# Export Institutional vs. Individual Data to CSV
if inst_foreign_table:
    df_inst_foreign.to_csv('foreign_data.csv', index=False, encoding='utf-8-sig')
    print("Saved Data")

# Export Block Trades Data to CSV
if block_trades_table:
    df_block_trades.to_csv('block_trades.csv', index=False, encoding='utf-8-sig')
    print("Saved Block Trades data")


Saved Data
Saved Block Trades data
